In [25]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_from_disk
from transformers import PreTrainedTokenizerFast

In [26]:

def load_tokenizers(base_dir: str = "NeuralMachineTranslation"):
    """
    Load the French (src) and English (tgt) BPE tokenizers
    from the provided tokenizer folders.
    """
    fr_path = f"{base_dir}/tokenizer_fr"
    en_path = f"{base_dir}/tokenizer_en"
 
    src_tokenizer = PreTrainedTokenizerFast.from_pretrained(fr_path)
    tgt_tokenizer = PreTrainedTokenizerFast.from_pretrained(en_path)
 
    # Ensure special tokens are set
    for tok in [src_tokenizer, tgt_tokenizer]:
        if tok.pad_token is None:
            tok.add_special_tokens({"pad_token": "<pad>"})
        if tok.bos_token is None:
            tok.add_special_tokens({"bos_token": "<s>"})
        if tok.eos_token is None:
            tok.add_special_tokens({"eos_token": "</s>"})
 
    print(f"FR tokenizer vocab size : {src_tokenizer.vocab_size}")
    print(f"EN tokenizer vocab size : {tgt_tokenizer.vocab_size}")
    print(f"PAD id  — FR: {src_tokenizer.pad_token_id} | EN: {tgt_tokenizer.pad_token_id}")
    print(f"BOS id  — EN: {tgt_tokenizer.bos_token_id}")
    print(f"EOS id  — EN: {tgt_tokenizer.eos_token_id}")
 
    return src_tokenizer, tgt_tokenizer
 

In [27]:

def inspect_dataset(base_dir: str = "NeuralMachineTranslation"):
    """
    Call this once to see what columns the Arrow dataset has.
    This tells us the exact field names for French and English.
    """
    dataset = load_from_disk(f"{base_dir}/parallel_en_fr_corpus")
    print(dataset)
    print("\nColumn names:", dataset["train"].column_names)
    print("\nFirst example:", dataset["train"][0])
    return dataset
 
 

In [28]:

class TranslationDataset(Dataset):
    """
    Wraps one split (train / validation / test) of the Arrow dataset.
 
    The Arrow dataset likely stores each example as:
        {"translation": {"en": "...", "fr": "..."}}
    OR as flat columns:
        {"en": "...", "fr": "..."}
 
    We handle both cases automatically.
    """
 
    def __init__(
        self,
        hf_split,                          # one HuggingFace dataset split
        src_tokenizer: PreTrainedTokenizerFast,
        tgt_tokenizer: PreTrainedTokenizerFast,
        src_lang: str = "text_fr",         # French = source
        tgt_lang: str = "text_en",         # English = target
        max_len: int = 32,
    ):
        self.max_len = max_len
        self.src_tokenizer = src_tokenizer
        self.tgt_tokenizer = tgt_tokenizer
 
        # ── Detect column layout ──────────────────────────────────────────
        cols = hf_split.column_names
        print(f"Dataset columns: {cols}")
 
        if "translation" in cols:
            # Nested format: {"translation": {"en": ..., "fr": ...}}
            src_sentences = [ex["translation"][src_lang] for ex in hf_split]
            tgt_sentences = [ex["translation"][tgt_lang] for ex in hf_split]
        elif src_lang in cols and tgt_lang in cols:
            # Flat format: {"en": ..., "fr": ...}
            src_sentences = list(hf_split[src_lang])
            tgt_sentences = list(hf_split[tgt_lang])
        else:
            raise ValueError(
                f"Cannot find '{src_lang}'/'{tgt_lang}' in columns: {cols}\n"
                f"Call inspect_dataset() to see the actual column names."
            )
 
        print(f"  Loaded {len(src_sentences)} sentence pairs.")
 
        # ── Tokenize everything up front ──────────────────────────────────
        self.src_ids = self._encode(src_tokenizer, src_sentences)
        self.tgt_ids = self._encode(tgt_tokenizer, tgt_sentences)
 
    def _encode(self, tokenizer, sentences):
        encoded = tokenizer(
            sentences,
            truncation=True,
            max_length=self.max_len,
            add_special_tokens=False,   # we add BOS/EOS manually below
        )
        return [torch.tensor(ids, dtype=torch.long) for ids in encoded["input_ids"]]
 
    def __len__(self):
        return len(self.src_ids)
 
    def __getitem__(self, idx):
        src = self.src_ids[idx]
 
        raw_tgt = self.tgt_ids[idx]
        bos = self.tgt_tokenizer.bos_token_id
        eos = self.tgt_tokenizer.eos_token_id
 
        # decoder_input : [BOS, t1, t2, ...]   — fed into the decoder
        # label         : [t1, t2, ..., EOS]   — what the model should predict
        decoder_input = torch.cat([torch.tensor([bos]), raw_tgt])[: self.max_len]
        label         = torch.cat([raw_tgt, torch.tensor([eos])])[: self.max_len]
 
  
        return src, decoder_input, label


In [29]:

def make_collate_fn(src_pad_id: int, tgt_pad_id: int):
    """
    Pads a batch of variable-length sequences.
 
    Returns:
        src_batch        (B, S) — padded French source
        dec_input_batch  (B, T) — padded decoder inputs  [BOS + tokens]
        label_batch      (B, T) — padded labels          [tokens + EOS]
        src_lengths      (B,)   — true source lengths (for LSTM packing)
        tgt_lengths      (B,)   — true target lengths
    """
    def collate_fn(batch):
        src_seqs, dec_inputs, labels = zip(*batch)
 
        src_lengths = torch.tensor([len(s) for s in src_seqs])
        tgt_lengths = torch.tensor([len(l) for l in labels])
 
        src_batch       = pad_sequence(src_seqs,   batch_first=True, padding_value=src_pad_id)
        dec_input_batch = pad_sequence(dec_inputs, batch_first=True, padding_value=tgt_pad_id)
        label_batch     = pad_sequence(labels,     batch_first=True, padding_value=tgt_pad_id)
 
        return src_batch, dec_input_batch, label_batch, src_lengths, tgt_lengths
 
    return collate_fn
 

In [30]:

def get_dataloaders(
    base_dir: str = "NeuralMachineTranslation",
    batch_size: int = 32,
    max_len: int = 32,
    num_workers: int = 2,
):
    """
    Full pipeline:
        load tokenizers → load Arrow dataset → build DataLoaders
 
    Returns:
        train_loader, val_loader, test_loader,
        src_tokenizer (FR), tgt_tokenizer (EN)
    """
    src_tok, tgt_tok = load_tokenizers(base_dir)
 
    raw = load_from_disk(f"{base_dir}/parallel_en_fr_corpus")
    print(f"\nDataset splits: {list(raw.keys())}")
 
    train_ds = TranslationDataset(raw["train"],      src_tok, tgt_tok, max_len=max_len)
    val_ds   = TranslationDataset(raw["validation"], src_tok, tgt_tok, max_len=max_len)
    test_ds  = TranslationDataset(raw["test"],       src_tok, tgt_tok, max_len=max_len)
 
    collate_fn = make_collate_fn(
        src_pad_id=src_tok.pad_token_id,
        tgt_pad_id=tgt_tok.pad_token_id,
    )
 
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_fn, num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              collate_fn=collate_fn, num_workers=num_workers, pin_memory=True)
 
    return train_loader, val_loader, test_loader, src_tok, tgt_tok
 

In [ ]:

if __name__ == "__main__":
    import sys
 
    base_dir = r"C:\Users\sara_\OneDrive - Alexandria University\Desktop\NLP\Assignment 1\NLP-Assignments\NeuralMachineTranslation"
 
    print("=" * 50)
    print("Step 1: Inspecting raw dataset...")
    print("=" * 50)
    inspect_dataset(base_dir)
 
    print("\n" + "=" * 50)
    print("Step 2: Building DataLoaders...")
    print("=" * 50)
    train_loader, val_loader, test_loader, src_tok, tgt_tok = get_dataloaders(
        base_dir=base_dir, batch_size=4, max_len=32
    )
 
    print("\n" + "=" * 50)
    print("Step 3: Checking one batch...")
    print("=" * 50)
    src, dec_input, labels, src_len, tgt_len = next(iter(train_loader))
    print(f"src shape        : {src.shape}")
    print(f"dec_input shape  : {dec_input.shape}")
    print(f"labels shape     : {labels.shape}")
    print(f"src_lengths      : {src_len}")
 
    print("\nDecoding first example:")
    print(f"  FR (src)  : {src_tok.decode(src[0].tolist(), skip_special_tokens=True)}")
    print(f"  EN (label): {tgt_tok.decode(labels[0].tolist(), skip_special_tokens=True)}")
    print("\nAll good!")

Step 1: Inspecting raw dataset...
DatasetDict({
    train: Dataset({
        features: ['text_en', 'text_fr'],
        num_rows: 8701
    })
    validation: Dataset({
        features: ['text_en', 'text_fr'],
        num_rows: 485
    })
    test: Dataset({
        features: ['text_en', 'text_fr'],
        num_rows: 486
    })
})

Column names: ['text_en', 'text_fr']

First example: {'text_en': 'i m tough .', 'text_fr': 'je suis dure .'}

Step 2: Building DataLoaders...
FR tokenizer vocab size : 3200
EN tokenizer vocab size : 3200
PAD id  — FR: 3 | EN: 3
BOS id  — EN: 1
EOS id  — EN: 2

Dataset splits: ['train', 'validation', 'test']
Dataset columns: ['text_en', 'text_fr']
  Loaded 8701 sentence pairs.
Dataset columns: ['text_en', 'text_fr']
  Loaded 485 sentence pairs.
Dataset columns: ['text_en', 'text_fr']
  Loaded 486 sentence pairs.

Step 3: Checking one batch...


C:\Users\sara_\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
